Se carga la base de datos

Se trabaja con el módulo auxiliar process_data, el cual define una carga personalizada para estos ejemplos de ataques a modelos fedrados. Para más detalles sobre como efectuar la carga de datos con Flex ir a la documentación correspondiente.

Este módulo permite la carga de dataset de procesamiento de imágenes como: Mnist, Fmnist, Cifar10 y Cifar100. Además del dataset tabular nursery.

In [36]:
from process_data import *
from copy import deepcopy

flex_data, server_id = load_and_preprocess_horizontal(dataname="mnist", trasnform=False, nodes=5)
adv_data = flex_data[server_id]

torch.Size([60000, 28, 28])


A continuación, se define la arquitectura de los modelos locales de los clientes. Para el presente ejemplo se trabaja con modelos neuronales de pytorch.

Se utiliza el módulo networks_models, quien contiene una serie de modelos neuronales auxiliares de pytorch, para el trabajo con las bases de datos anteriormente mencionadas. Además se utiliza el módulo auxiliar networks_execution, que define la ejecución del entrenamiento y otros detalles de estos modelos.

Para establecer un modelo personalizado, ir a la documentación de Flex.

In [37]:
from networks_models import *
from networks_execution import *
from flex.pool import init_server_model
from flex.model import FlexModel

device = (
    "cuda"
    if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available() else "cpu"
)

net_config = ExecutionNetwork()

@init_server_model
def build_server_model():
    server_flex_model = FlexModel()

    criterion, model, optimizer = net_config.for_fd_server_model_config()

    server_flex_model["model"] = model.to(device)
    # Required to store this for later stages of the FL training process
    server_flex_model["criterion"] = criterion
    server_flex_model["optimizer_func"] = optimizer
    server_flex_model["optimizer_kwargs"] = {}

    return server_flex_model

Se define la arquitectura del modelo federado

In [38]:
from flex.pool import FlexPool
clients = 2

pool = FlexPool.client_server_pool(
        fed_dataset= flex_data, server_id=server_id, init_func = build_server_model
    )

#selected_test_clients_pool = pool.clients.select(clients)
#selected_test_clients = selected_test_clients_pool.clients

Se define la función para desplegar el modelo global en cada cliente

In [39]:
from flex.pool.decorators import (  # noqa: E402
    deploy_server_model,
)

@deploy_server_model
def deploy_serv(server_flex_model: FlexModel): 

    new_model = deepcopy(server_flex_model)

    return new_model

#pool.servers.map(deploy_serv, selected_test_clients)

Se define la ronda de entrenamiento local de un cliente, empleando el módulo networks_execution

In [40]:
def train(client_flex_model: FlexModel, client_data: Dataset):

    train_dataset = client_data.to_torchvision_dataset(transform = mnist_transform())
    client_dataloader = DataLoader(train_dataset, batch_size = 256)

    model = client_flex_model["model"]
    model = model.to(device)

    client_flex_model["previous_model"] = deepcopy(model)
    optimizer = client_flex_model["optimizer_func"]
    criterion = client_flex_model["criterion"]

    net_config.train_network(local_epochs = 1, criterion = criterion, optimizer = optimizer,
                            momentum = 0.9, lr = 0.005, trainloader = client_dataloader, testloader= None, 
                            model=model)
    
    return client_flex_model

#selected_test_clients.map(train)

Se efectúa la agregación del modelo federado

In [41]:
from flex.pool import collect_client_diff_weights_pt
from flex.pool import fed_avg
from flex.pool import set_aggregated_diff_weights_pt


#pool.aggregators.map(collect_client_diff_weights_pt, selected_test_clients)
#pool.aggregators.map(fed_avg)
#pool.aggregators.map(set_aggregated_diff_weights_pt, pool.servers)

Se define el ataque de inferencia de inversión de gradientes. Se definen 3 versiones. 

1- Toma los gradientes de los clientes y el modelo global para reconstruir imágenes. No recibe buenos resultados

2- Se ubica en el servidor, el atacante tiene conocimiento del modelo global del servidor y lo utiliza para reconstruir imágenes a partir de datos conocidos

3- El atacante intercepta las actualizaciones de cada cliente y reconstruye imágenes (Una por una),a partir de datos conocidos

En esta reconstrucción es importante definir los valores de media y desviación estandar u otro patrón o rango de posible valores de los píxeles

In [42]:
from attack.my_models_attacks.inversed_gradient import *
from flexclash.model import model_inference_known_a_model_information, model_inference_known_agregator_information
import traceback

import matplotlib.pyplot as plt 
import torchvision.transforms as transforms



_, global_model, _ = net_config.for_fd_server_model_config()
attack = reconstruction_gradient(global_model = global_model)

client_index = dict()
round = 0

def find_client_id(index):
    for key, value in client_index.items():
        if value == index:
            return key
    return None

@model_inference_known_a_model_information
def inferencer(client_model: FlexModel):#Ver lo de la perdida que está dando Nan, puede ser que estoy solo viendo pocas iteraciones de todo y nada se a optimizado
                                                             #Ver bien lo de la media y el std, ver que se escojan los clientes para infestar y que sea a partir de un número de iteraciones
    
    try:
        train_dataset = adv_data.to_torchvision_dataset(transform = mnist_transform())
        mean, std = attack.get_meanstd(train_dataset)
        mean = torch.as_tensor(mean,)[:, None, None]
        std = torch.as_tensor(std,)[:, None, None]
        labels = [torch.as_tensor((label,)) for label in np.array(train_dataset.data.y_data)]
    except Exception as e:
        print(f"Error: {e.__class__.__name__} - {e}")
        tb = traceback.format_exc()
        print(tb)

    num_images = 2
    labels = torch.cat(labels)
    labels=labels.long()
    attack.set_config_attack(lr = 1, restarts = 1, max_iter = 100)

    try:
        output, stats = attack.reconstruction_gradient_attack(client_model["model"], deepcopy(client_model["previous_model"]), train_dataset[0][0].shape, mean, std,
                                                                     num_images = num_images, labels = None)
        count = 0
        countN = 0
        
        for i in range(len(train_dataset)):
            print(train_dataset[i][0].size())
            img = train_dataset[i][0]
            image = img.squeeze()
            transform_to_pil = transforms.ToPILImage()
            image_pil = transform_to_pil(image)
            plt.imshow(image_pil, cmap='gray')
            plt.title("Imagen en Escala de Grises")
            plt.axis('off')
            plt.show()
            countN += 1
            if countN == 10:
                break

        print("Salidas:")
        for img in output:
            image = img.squeeze()
            transform_to_pil = transforms.ToPILImage()
            image_pil = transform_to_pil(image)
            plt.imshow(image_pil, cmap='gray')
            plt.title("Imagen en Escala de Grises")
            plt.axis('off')
            plt.show()
            count+=1
            if count == 10:
                break
    except Exception as e:
        print(f"Error: {e.__class__.__name__} - {e}")
        tb = traceback.format_exc()
        print(tb)


@model_inference_known_a_model_information
def inferencer2(server_model: FlexModel):#Ver lo de la perdida que está dando Nan, puede ser que estoy solo viendo pocas iteraciones de todo y nada se a optimizado
                                                             #Ver bien lo de la media y el std, ver que se escojan los clientes para infestar y que sea a partir de un número de iteraciones

    try:
        train_dataset = adv_data.to_torchvision_dataset(transform = mnist_transform())
        mean, std = attack.get_meanstd(train_dataset)
        #mean = torch.as_tensor(mean,)[:, None, None]
        #std = torch.as_tensor(std,)[:, None, None]
        mean = torch.as_tensor([0.5],)[:, None, None]
        std = torch.as_tensor([0.5],)[:, None, None]
        labels = [torch.as_tensor((label,)) for label in np.array(train_dataset.data.y_data)]
    except Exception as e:
        print(f"Error: {e.__class__.__name__} - {e}")
        tb = traceback.format_exc()
        print(tb)

    num_images = 5
    labels = torch.cat(labels)
    labels = labels.long()

    attack.set_config_attack(lr = 1, restarts = 1, max_iter = 100)

    try:
        output, stats = attack.reconstruction_one_by_one(data_adv = train_dataset, 
                                                              server_model = server_model["model"], 
                                                              local_lr = 0.001, local_steps = 1,
                                                              dim_imgs = train_dataset[0][0].shape, 
                                                              mean = mean, std = std, num_images = num_images)
        
        attack.save_in_server_summary(images = output, stats = stats, round = round)
        print("Save server",round)
    except Exception as e:
        print(f"Error: {e.__class__.__name__} - {e}")
        tb = traceback.format_exc()
        print(tb)

@model_inference_known_agregator_information
def inferencer3(list_of_wigths_clients: list):#Ver lo de la perdida que está dando Nan, puede ser que estoy solo viendo pocas iteraciones de todo y nada se a optimizado
                                                             #Ver bien lo de la media y el std, ver que se escojan los clientes para infestar y que sea a partir de un número de iteraciones

    try:
        train_dataset = adv_data.to_torchvision_dataset(transform = mnist_transform())
        mean, std = attack.get_meanstd(train_dataset)
        #mean = torch.as_tensor(mean,)[:, None, None]
        #std = torch.as_tensor(std,)[:, None, None]
        mean = torch.as_tensor([0.5],)[:, None, None]
        std = torch.as_tensor([0.5],)[:, None, None]
        labels = [torch.as_tensor((label,)) for label in np.array(train_dataset.data.y_data)]
    except Exception as e:
        print(f"Error: {e.__class__.__name__} - {e}")
        tb = traceback.format_exc()
        print(tb)

    num_images = 5
    labels = torch.cat(labels)
    labels = labels.long()

    for client in range(len(list_of_wigths_clients)):
        print("Client Img, reconstruct")
        attack.set_config_attack(list_of_wigths_clients[client] ,lr = 1, restarts = 1, max_iter = 100)

        try:
            output, stats = attack.reconstruction_one_by_one(data_adv = train_dataset, server_model = attack.model,
                                                                local_lr = 0.001, local_steps = 1,
                                                                dim_imgs = train_dataset[0][0].shape, 
                                                                mean = mean, std = std, num_images = num_images)
            client_id = find_client_id(client)
            attack.save_in_clients_summary(client = client_id, stats = stats, round = round, images = output)
            print(round)
        except Exception as e:
            print(f"Error: {e.__class__.__name__} - {e}")
            tb = traceback.format_exc()
            print(tb)


Se evalúa el modelo federado

In [43]:
def evaluate_global_model(server_flex_model: FlexModel, test_data: Dataset):#falta poner esto
    model = server_flex_model["model"]
    model.eval()
    test_loss = 0
    test_acc = 0
    total_count = 0
    model = model.to(device)
    criterion = server_flex_model["criterion"]
    # get test data as a torchvision object
    test_dataset = test_data.to_torchvision_dataset(transform = mnist_transform())
    test_dataloader = DataLoader(
        test_dataset, batch_size=256, shuffle=True, pin_memory=False
    )
    losses = []
    with torch.no_grad():
        for data, target in tqdm(test_dataloader):
            total_count += target.size(0)
            data, target = data.to(device), target.to(device)
            output = model(data)
            losses.append(criterion(output, target).item())
            pred = output.data.max(1, keepdim=True)[1]
            test_acc += pred.eq(target.data.view_as(pred)).long().cpu().sum().item()

    test_loss = sum(losses) / len(losses)
    test_acc /= total_count
    return test_loss, test_acc

#metrics = pool.servers.map(evaluate_global_model)

#loss, acc = metrics[0]
#print(f"Server: Test acc: {acc:.4f}, test loss: {loss:.4f}")

Se muestran los resultados del ataque. En este caso solo se muestran las imágenes reconstruídas y las medidas del ataque atandiendo a:

1- Similitud de cada imagen reconstruida, atendiendo a similitud de coseno por cada capa del modelo adversario empleado en la inferencia. Los valores de la última y penúltima capa indican que tan similar es la imagen a la real

2- Suma de valores residuales, mide que tan bien es la predicción dada o la imagen reconstruída respecto a una predicción o reconstrucción correcta, en cada capa del modelo adversario. En concreto mide el error del modelo adversario. Esta variante de la medición es una suma.

3- Media de valores residuales, mide que tan bien es la predicción dada o la imagen reconstruída respecto a una predicción o reconstrucción correcta, en cada capa del modelo adversario. En concreto mide el error del modelo adversario. Esta variante de la medición es una media.

Se pretende contar con un validador que indque si los reconstruido es una imagen o no, a partir de un dicriminador previamente entrenado

In [44]:
def show_imgs_and_stats_for_clients():
    dict_imgs_for_clients = attack.summary_clients_imgs
    dict_stats_for_clients = attack.summary_clients_sts
    test_adv_dataset = adv_data.to_torchvision_dataset(transform = mnist_transform())
    
    datas_x_clients_rounds = dict()

    print("Information, images:")
    for act_round in dict_imgs_for_clients.keys():
        print("Round:", act_round)
        datas_x_clients_rounds[act_round] = dict()
        for client in dict_imgs_for_clients[act_round].keys():
            imgs_cient = dict_imgs_for_clients[act_round][client]
            print("Client:", client)
            data = attack.evaluation_metrics(imgs = imgs_cient, data_adv = test_adv_dataset)
            show_metrics_results(data)
            datas_x_clients_rounds[act_round][client] = data
            for img in range(len(imgs_cient)):
                print("Image:", img+1)
                img_to_show = imgs_cient[img]
                image = img_to_show.squeeze()
                transform_to_pil = transforms.ToPILImage()
                image_pil = transform_to_pil(image)
                plt.imshow(image_pil, cmap='gray')
                plt.title("Imagen en Escala de Grises")
                plt.axis('off')
                plt.show()
    
    print("Information, stats:")
    for act_round in dict_stats_for_clients.keys():
        print("Round:", act_round)
        for client in dict_stats_for_clients[act_round].keys():
            stats_cient = dict_stats_for_clients[act_round][client]
            print("Client:", client)
            for stats_for_img in range(len(stats_cient)):
                print("Image:", stats_for_img + 1)
                print("Stats values:", stats_cient[stats_for_img])



def show_imgs_and_stats_for_server():
    dict_imgs = attack.summary_server_inf_img
    dict_stats = attack.summary_server_inf_sts
    test_adv_dataset = adv_data.to_torchvision_dataset(transform = mnist_transform())
    
    datas_x_rounds = dict()

    print("Information, images:")
    for act_round in dict_imgs.keys():
        print("Round:", act_round)
        imgs = dict_imgs[act_round]
        data = attack.evaluation_metrics(imgs = imgs, data_adv = test_adv_dataset)
        show_metrics_results(data)
        datas_x_rounds[act_round] = data
        for img in range(len(imgs)):
            print("Image:", img+1)
            img_to_show = imgs[img]
            image = img_to_show.squeeze()
            transform_to_pil = transforms.ToPILImage()
            image_pil = transform_to_pil(image)
            plt.imshow(image_pil, cmap='gray')
            plt.title("Imagen en Escala de Grises")
            plt.axis('off')
            plt.show()
    
    print("Information, stats:")
    for act_round in dict_stats.keys():
        print("Round:", act_round)
        stats_round = dict_stats[act_round]
        for stats_for_img in range(len(stats_round)):
                print("Image:", stats_for_img + 1)
                print("Stats values:", stats_round[stats_for_img])


def show_metrics_results(data_metric):
    num_img = 1
    for data in data_metric:
        #print("Para la imagen:", num_img)
        #print(type(data), data)
        fig, axes = plt.subplots(1, 3, sharey=False, figsize=(18,6))
        axes[0].semilogy(list(data['se'].values())[:-1])
        axes[0].set_title('SE')
        axes[1].semilogy(list(data['mse'].values())[:-1])
        axes[1].set_title('MSE')
        axes[2].semilogy(list(data['sim'].values())[:-1])
        axes[2].set_title('Similarity')
        plt.tight_layout()
        plt.show()
        print("-----------------------------------------------------------")
        num_img+=1

Para limpiar los modelos en memoria. Opcional

In [45]:
def clean_up_models(client_model: FlexModel, _):
    import gc

    client_model.clear()
    gc.collect()

Se definen las rondas de entrenamiento del modelo federado. En este caso se debe tener en cuenta dos versiones del ataque, cuando actúa sobre el agregador y cuando lo hace sobre el servidor. En ambas se debe tener en cuenta la ronda actual de entrenamiento y los índices de los cleintes, para tener un registro del ataque, por cada ronda y cada cliente.

In [46]:
def train_n_rounds(n_rounds, clients_per_round = 20):

    pool = FlexPool.client_server_pool(
        fed_dataset= flex_data, server_id=server_id, init_func=build_server_model
    )

    global round
    global client_index

    for i in range(n_rounds):
        round += 1
        print(f"\nRunning round: {i+1} of {n_rounds}")
        selected_clients_pool = pool.clients.select(clients_per_round)
        selected_clients = selected_clients_pool.clients
        pool.servers.map(deploy_serv, selected_clients)
        selected_clients.map(train)
        pool.aggregators.map(collect_client_diff_weights_pt, selected_clients)
        if i != 0:
            pos_c_list=0
            client_index = dict()
            for i in selected_clients.actor_ids:
                print("Cliente id:", i)
                client_index[i] = pos_c_list
                pos_c_list+=1
            pool.aggregators.map(inferencer3)
        pool.aggregators.map(fed_avg)
        pool.aggregators.map(set_aggregated_diff_weights_pt, pool.servers)
        metrics = pool.servers.map(evaluate_global_model)
        if round == n_rounds:
            pool.servers.map(inferencer2)
        selected_clients.map(clean_up_models)
        loss, acc = metrics[0]
        print(f"Server: Test acc: {acc:.4f}, test loss: {loss:.4f}")
    show_imgs_and_stats_for_clients()
    show_imgs_and_stats_for_server()

In [47]:
train_n_rounds(4, clients_per_round = 2)


Running round: 1 of 4


100%|██████████| 47/47 [00:05<00:00,  8.41it/s]


Después de la modif antes de fedavg tensor(1.0409)



A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.1.3 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "C:\Users\Adrian\PycharmProjects\pythonProject\.venv\Lib\site-packages\ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "C:\Users\Adrian\PycharmProjects\pythonProject\.venv\Lib\site-packages\traitlets\config\application.py", line 1075, in launch_instance
    app.start()
  File "C:\Users\Adrian\PycharmProjects\pythonProject\.venv\Lib\site-packages\ipykernel\kernelapp.py", line 739,

AttributeError: _ARRAY_API not found

...

AttributeError: _ARRAY_API not found

ImportError: numpy.core._multiarray_umath failed to import

100%|██████████| 40/40 [00:02<00:00, 16.06it/s]


Server: Test acc: 0.6765, test loss: 1.1357

Running round: 3 of 4


 45%|████▍     | 21/47 [00:02<00:03,  8.52it/s]

KeyboardInterrupt

